# Lab A: Instrument the Research Team with LangFuse

A trace is the recording of one end-to-end run of your system: a tree of **spans**, each
covering one unit of work (an agent turn, a tool call, an LLM generation), with a start time,
an end time, and whatever inputs/outputs you choose to attach. Without tracing, a multi-agent
failure shows you only the final state; with it, you can see **which** agent produced the bad
value and **how long** each step took. This lab wraps the Day 3 research team — the same
five-specialist-plus-supervisor graph you built last session — with nested LangFuse spans, tags
each run with a session/user id, and builds a small per-agent cost/latency view from what the
trace captured.

| Part | You build | Key idea |
|---|---|---|
| **A1** | A `traced()` span decorator | A span is a labelled duration with inputs/outputs, nothing more |
| **A2** | The traced team, tagged per run | Nesting is automatic — it follows the call stack |
| **A3** | A per-agent cost/latency table | The trace *is* the data source for a dashboard |
| **A4** | Traced vs. untraced timing | Tracing has overhead — never let it stand in for real latency |

**TA note carried over from the spec:** tracing overhead can mislead learners about "real" performance. Part A4 measures that overhead directly so it stops being a surprise.

## Setup

## Get Langfuse Cloud keys

1. Go to [cloud.langfuse.com](https://cloud.langfuse.com/) and sign up (email, Google, or GitHub).
2. Create a project (or open an existing one).
3. Go to **Project Settings → API Keys → Create new API credentials**.
4. Copy both values immediately — the **secret key** is shown only once:
   - Public key — starts with `pk-lf-...`
   - Secret key — starts with `sk-lf-...`
5. Note your region's base URL:
   - EU (default) [https://cloud.langfuse.com](https://cloud.langfuse.com),
   - US [https://us.cloud.langfuse.com](https://us.cloud.langfuse.com),
   - Self-hosted: use your instance's URL.
6. Run the cell below to generate a `.env`, paste your keys in, save, then re-run.

In [ ]:
# === Guided setup ===
%pip install -q --break-system-packages "langfuse==4.14.0" "langgraph==1.2.9" "litellm==1.93.0" "python-dotenv==1.2.2" "pandas==3.0.3" "tenacity==9.1.4"

import os

# Edit these with your real keys, then re-run this cell.
LANGFUSE_SECRET_KEY="sk-lf-ffe3c346-a9b7-483a-9ff3-4c5c8da34e62"
LANGFUSE_PUBLIC_KEY="pk-lf-f495090d-0cd2-4e76-9857-a73954515299"
LANGFUSE_BASE_URL="https://us.cloud.langfuse.com"
GROQ_API_KEY         = "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"

with open(".env", "w") as f:
    f.write(f'LANGFUSE_PUBLIC_KEY="{LANGFUSE_PUBLIC_KEY}"\n')
    f.write(f'LANGFUSE_SECRET_KEY="{LANGFUSE_SECRET_KEY}"\n')
    f.write(f'LANGFUSE_BASE_URL="{LANGFUSE_BASE_URL}"\n')
    f.write(f'GROQ_API_KEY="{GROQ_API_KEY}"\n')

from dotenv import load_dotenv
load_dotenv(override=True)

from langfuse import get_client
langfuse = get_client()
if os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"):
    print(f"Langfuse auth_check: {'ok' if langfuse.auth_check() else 'FAILED - check your keys'}")
else:
    print("Langfuse keys still empty - every cell below still runs; traces just won't reach the Langfuse UI yet.")


In [ ]:
# Readiness check - confirm every moving part imports BEFORE you build on it.
from dotenv import load_dotenv
load_dotenv()  # reads LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY from a .env, if present

import sys, os
checks = {}
for label, imp in [
    ("langfuse",  "from langfuse import get_client"),
    ("langgraph", "from langgraph.graph import StateGraph, START, END"),
    ("litellm (optional LLM)", "import litellm"),
    ("pandas",    "import pandas as pd"),
]:
    try:
        exec(imp)
        checks[label] = "ok"
    except Exception as e:
        checks[label] = f"FAILED: {type(e).__name__}: {e}"

print("Environment readiness")
print("---------------------")
for k, v in checks.items():
    print(f"  {k:>24} : {v}")

from langfuse import get_client
langfuse = get_client()
if os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"):
    ok = langfuse.auth_check()
    print(f"  {'Langfuse auth_check':>24} : {'ok' if ok else 'FAILED - check your keys'}")
else:
    print(f"  {'Langfuse auth_check':>24} : SKIPPED - no LANGFUSE_PUBLIC_KEY/SECRET_KEY in .env")
    print("    Add them to the shared .env.template if you want traces to appear in the")
    print("    Langfuse UI. The SDK never raises on a missing/bad key (errors are caught and")
    print("    logged internally) - every cell below still runs and the local dashboard in")
    print("    A3 still works from data captured in this notebook.")

"""
EXPECTED OUTPUT
---------------
Environment readiness
----------------------
        langfuse : ok
       langgraph : ok
litellm (optional LLM) : ok
          pandas : ok
    Langfuse auth_check : SKIPPED - no LANGFUSE_PUBLIC_KEY/SECRET_KEY in .env
    (or "ok" if you configured a .env)
"""


---
## Recap · the Day 3 research team, solved

You built this graph in Day 3 Session 2: a **supervisor** routes work to five specialists
(planner, researcher, writer, fact-checker, reviewer); each specialist writes only to its own
state keys (`scoped()` enforces that); the fact-checker catches a planted fabricated citation
and the writer revises once, within a `MAX_REVISIONS` budget. The cell below is that solved
team, pasted in full so this lab is self-contained regardless of how your Day 3 notebook turned
out. Nothing here changes today — you are only wrapping it with tracing.


In [ ]:
# Provided — the Day 3 research team, solved (read, run, no changes needed).
from typing import TypedDict, Annotated
from operator import add
import functools, inspect, re
from langgraph.graph import StateGraph, START, END

MAX_REVISIONS = 2

class TeamState(TypedDict):
    brief: str
    plan: list[str]
    findings: Annotated[list[dict], add]
    draft: str
    fact_check: dict
    review: dict
    revision_count: int
    next_agent: str
    status: str
    log: Annotated[list[str], add]

AGENT_SCOPES = {
    "planner":      {"plan"},
    "researcher":   {"findings"},
    "writer":       {"draft", "revision_count", "fact_check", "review"},
    "fact_checker": {"fact_check"},
    "reviewer":     {"review"},
    "supervisor":   {"next_agent"},
    "escalate":     {"status"},
}

def scoped(role: str):
    allowed = AGENT_SCOPES[role] | {"log"}
    def decorator(fn):
        if inspect.iscoroutinefunction(fn):
            @functools.wraps(fn)
            async def awrapper(state):
                result = await fn(state)
                extra = set(result) - allowed
                if extra:
                    raise PermissionError(
                        f"agent '{role}' wrote outside its scope: {sorted(extra)}; allowed={sorted(allowed)}")
                return result
            return awrapper
        @functools.wraps(fn)
        def wrapper(state):
            result = fn(state)
            extra = set(result) - allowed
            if extra:
                raise PermissionError(
                    f"agent '{role}' wrote outside its scope: {sorted(extra)}; allowed={sorted(allowed)}")
            return result
        return wrapper
    return decorator

import csv
ROWS = [
 ("S1","vendor_concentration","Payments vendor register",
  "Ninety-one percent of card transactions route through a single processor, Northgate Pay."),
 ("S2","vendor_concentration","Contract exposure summary",
  "The Northgate Pay master agreement auto-renews annually with a ninety-day termination notice."),
 ("S3","incident_history","Incident post-mortem 2025-11",
  "A four-hour Northgate Pay outage in November 2025 halted eighty-eight percent of checkout volume."),
 ("S4","mitigation","Failover design note",
  "A secondary processor can be integrated behind the existing payment abstraction in about six weeks."),
 ("S5","mitigation","Cost model",
  "Maintaining a warm secondary processor adds an estimated forty thousand per year in fixed fees."),
 ("S6","contract_terms","Procurement standard",
  "Any vendor above sixty percent of transaction volume requires an approved concentration waiver."),
]
with open("knowledge_base.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh); w.writerow(["id", "topic", "title", "text"]); w.writerows(ROWS)
with open("knowledge_base.csv", newline="", encoding="utf-8") as fh:
    KB = list(csv.DictReader(fh))

def search_kb(topic: str) -> list[dict]:
    return [r for r in KB if r["topic"] == topic]

BRIEF = ("Assess our vendor concentration risk in the payments stack: how exposed are we, "
         "what has gone wrong before, and what could we do about it?")

TOPIC_KEYWORDS = {
    "vendor_concentration": ["concentration", "exposed", "dependency"],
    "incident_history":     ["gone wrong", "outage", "incident"],
    "mitigation":           ["do about it", "mitigat", "options"],
    "contract_terms":       ["contract", "waiver"],
}

def context_for(role: str, state: TeamState) -> dict:
    if role == "planner":      return {"brief": state["brief"]}
    if role == "researcher":   return {"plan": state["plan"], "already": [f["topic"] for f in state["findings"]]}
    if role == "writer":       return {"brief": state["brief"], "plan": state["plan"],
                                       "findings": state["findings"], "problems": state["fact_check"]}
    if role == "fact_checker": return {"draft": state["draft"], "findings": state["findings"]}
    if role == "reviewer":     return {"draft": state["draft"], "plan": state["plan"]}
    return {}

@scoped("planner")
def planner_node(state: TeamState) -> dict:
    brief = context_for("planner", state)["brief"].lower()
    plan = [topic for topic, kws in TOPIC_KEYWORDS.items() if any(k in brief for k in kws)]
    return {"plan": plan, "log": [f"planner: plan has {len(plan)} step(s) -> {plan}"]}

def compose(state: TeamState, drop_tags=()) -> str:
    out = [f"# Briefing: {state['brief'][:46]}...", ""]
    for topic in state["plan"]:
        out.append(f"## {topic.replace('_', ' ').title()}")
        for f in [x for x in state["findings"] if x["topic"] == topic]:
            out.append(f"{f['text']} [{f['id']}]")
        out.append("")
    out.append("## Risks")
    out.append("Concentration on a single processor is the dominant risk. [S1]")
    out.append("Northgate Pay is the regional market leader and unlikely to fail. [S9]")
    text = "\n".join(out)
    if drop_tags:
        text = "\n".join(l for l in text.split("\n") if not any(f"[{t}]" in l for t in drop_tags))
    return text

@scoped("writer")
def writer_node(state: TeamState) -> dict:
    ctx = context_for("writer", state)
    problems = ctx["problems"].get("unsupported", []) if ctx["problems"] else []
    is_revision = bool(state["draft"])
    draft = compose(state, drop_tags=problems)
    return {"draft": draft,
            "revision_count": state["revision_count"] + (1 if is_revision else 0),
            "fact_check": {}, "review": {},
            "log": [f"writer: {'revision ' + str(state['revision_count'] + 1) if is_revision else 'draft v0'}"
                    f" ({len(draft.split())} words, dropped={problems})"]}

@scoped("researcher")
def researcher_node(state: TeamState) -> dict:
    ctx = context_for("researcher", state)
    todo = [t for t in ctx["plan"] if t not in ctx["already"]]
    topic = todo[0]
    hits = search_kb(topic)
    findings = [{"topic": topic, "id": h["id"], "text": h["text"]} for h in hits]
    return {"findings": findings, "log": [f"researcher: retrieved {len(findings)} record(s) for '{topic}'"]}

@scoped("fact_checker")
def fact_checker_node(state: TeamState) -> dict:
    ctx = context_for("fact_checker", state)
    cited = set(re.findall(r"\[(S\d+)\]", ctx["draft"]))
    supported = {f["id"] for f in ctx["findings"]}
    unsupported = sorted(cited - supported)
    ok = not unsupported
    return {"fact_check": {"ok": ok, "unsupported": unsupported},
            "log": [f"fact_checker: {'all citations supported' if ok else str(len(unsupported)) + ' unsupported: ' + ','.join(unsupported)}"]}

@scoped("reviewer")
def reviewer_node(state: TeamState) -> dict:
    ctx = context_for("reviewer", state)
    notes = []
    for topic in ctx["plan"]:
        if f"## {topic.replace('_', ' ').title()}" not in ctx["draft"]:
            notes.append(f"missing section: {topic}")
    if "## Risks" not in ctx["draft"]:
        notes.append("missing Risks section")
    if len(ctx["draft"].split()) > 300:
        notes.append(f"too long: {len(ctx['draft'].split())} words")
    approved = not notes
    return {"review": {"approved": approved, "notes": notes},
            "log": [f"reviewer: {'approved' if approved else 'changes requested: ' + '; '.join(notes)}"]}

@scoped("escalate")
def escalate_node(state: TeamState) -> dict:
    return {"status": "escalated_to_human",
            "log": [f"ESCALATED after {state['revision_count']} revision(s) — human review required"]}

def supervisor_policy(state: TeamState) -> str:
    if not state["plan"]:
        return "planner"
    if set(state["plan"]) - {f["topic"] for f in state["findings"]}:
        return "researcher"
    if not state["draft"]:
        return "writer"
    if not state["fact_check"]:
        return "fact_checker"
    if not state["fact_check"]["ok"]:
        return "writer" if state["revision_count"] < MAX_REVISIONS else "escalate"
    if not state["review"]:
        return "reviewer"
    if not state["review"]["approved"]:
        return "writer" if state["revision_count"] < MAX_REVISIONS else "escalate"
    return "done"

@scoped("supervisor")
def supervisor_node(state: TeamState) -> dict:
    nxt = supervisor_policy(state)
    return {"next_agent": nxt, "log": [f"supervisor -> {nxt}"]}

SPECIALISTS = ["planner", "researcher", "writer", "fact_checker", "reviewer"]
NODE_FNS = {"supervisor": supervisor_node, "planner": planner_node, "researcher": researcher_node,
            "writer": writer_node, "fact_checker": fact_checker_node, "reviewer": reviewer_node,
            "escalate": escalate_node}

def build_team(node_fns):
    """Wire the star topology from a {name: fn} mapping - lets Lab A swap in traced nodes
    without touching the topology, exactly like Day 3's MCP swap changed only one node."""
    tb = StateGraph(TeamState)
    for name, fn in node_fns.items():
        tb.add_node(name, fn)
    tb.add_edge(START, "supervisor")
    tb.add_conditional_edges("supervisor", lambda s: s["next_agent"],
                             {**{a: a for a in SPECIALISTS}, "escalate": "escalate", "done": END})
    for a in SPECIALISTS:
        tb.add_edge(a, "supervisor")
    tb.add_edge("escalate", END)
    return tb.compile()

team = build_team(NODE_FNS)
seed = {"brief": BRIEF, "plan": [], "findings": [], "draft": "", "fact_check": {},
        "review": {}, "revision_count": 0, "next_agent": "", "status": "", "log": []}

untraced_final = team.invoke(seed, {"recursion_limit": 50})
print(f"Untraced sanity run: {len(untraced_final['log'])} trajectory lines, "
      f"converged={untraced_final['status']  == ''}")

"""
EXPECTED OUTPUT
---------------
Untraced sanity run: 19 trajectory lines, converged=True
"""


---
## A1 · A span decorator

Every span needs a name, a start/end time, and (optionally) recorded input/output. You will
write one small decorator, `traced(role)`, that wraps any team node function so calling it opens
a Langfuse span named after the role, records the state keys it read and wrote, and closes the
span when the node returns — including on exception, so a crashing agent still shows up in the
trace instead of silently vanishing.

Also keep a **local** record of every call (`RUN_EVENTS`) — role, duration in milliseconds, and
whether it errored. That local list is what A3's dashboard reads from; it does not require
pulling data back out of Langfuse.


In [ ]:
# TODO A1 — the span decorator.
import time

RUN_EVENTS = []   # each entry: {"role": str, "ms": float, "ok": bool}

def traced(role: str):
    """Wrap a node function so every call opens a named Langfuse span, records
    input/output on it, and appends a timing record to RUN_EVENTS."""
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(state):
            t0 = time.perf_counter()
            ok = True
            try:
                with langfuse.start_as_current_observation(as_type="span", name=f"agent:{role}") as span:
                    result = fn(state)
                    span.update(
                        input={k: state[k] for k in ("plan", "revision_count") if k in state},
                        output={"wrote": sorted(set(result) - {"log"}),
                                "log": result.get("log", [])},
                    )
                    return result
            except Exception:
                ok = False
                raise
            finally:
                RUN_EVENTS.append({"role": role, "ms": (time.perf_counter() - t0) * 1000, "ok": ok})
        return wrapper
    return decorator

# Smoke-test on a dummy node, inside a root span (so nesting has something to nest under).
@traced("dummy")
def dummy_node(state):
    return {"log": ["dummy ran"]}

RUN_EVENTS.clear()
with langfuse.start_as_current_observation(as_type="span", name="smoke-test"):
    dummy_node({})
langfuse.flush()

print(f"RUN_EVENTS after smoke test: {RUN_EVENTS}")

"""
EXPECTED OUTPUT
---------------
RUN_EVENTS after smoke test: [{'role': 'dummy', 'ms': <some float>, 'ok': True}]
"""


In [ ]:
# Self-check A1 — the decorator must run the node AND record a timing event, even on failure.
assert len(RUN_EVENTS) == 1, "expected exactly one event from the smoke test"
ev = RUN_EVENTS[0]
assert ev["role"] == "dummy" and ev["ok"] is True and ev["ms"] >= 0

@traced("boom")
def boom_node(state):
    raise ValueError("simulated node failure")

RUN_EVENTS.clear()
try:
    with langfuse.start_as_current_observation(as_type="span", name="smoke-test-2"):
        boom_node({})
    raise AssertionError("the exception should have propagated")
except ValueError:
    pass
langfuse.flush()

assert RUN_EVENTS and RUN_EVENTS[0]["ok"] is False, "a failing node must still record ok=False"
print("PASS — traced() records both successful and failing calls, and never swallows the error.")

"""
EXPECTED OUTPUT
---------------
PASS — traced() records both successful and failing calls, and never swallows the error.
"""


---
## A1b · Nested spans (prof's note, part 1)

**Concept:** a span can contain other spans — that's what makes a Langfuse trace a *tree*, not a
flat list. The "One Request, Forty Spans" slide showed this: `retrieval_agent.run` had its own
child spans for `vector_search` and `rerank`, so a slow or bad retrieval step was visible on its
own, not hidden inside one big number for the whole agent.

In this lab, `researcher_node`'s retrieval step is the call to `search_kb`. Right now A1's
`traced("researcher")` decorator gives you exactly one span per researcher call — you can see
*that* it ran and *how long it took*, but not what happened inside it. Opening a second,
**nested** span around just the `search_kb` call — using `langfuse.start_as_current_observation`
again, from inside code that's already running inside the outer span — records the retrieval step
as its own child span, with its own input (the search topic) and output (how many hits it found).
Nesting happens automatically: because the code runs on the same call stack as the outer span,
Langfuse attaches the new span as its child without you passing any span object around by hand —
the same mechanism A2's markdown cell describes for the researcher/writer/etc. spans themselves.


In [ ]:
# TODO A1b (prof's note) - nested span for retrieval, inside the researcher's own span.
# Ref: "One Request, Forty Spans" slide - vector_search/rerank nested under retrieval_agent.run.
@scoped("researcher")
def researcher_node(state: TeamState) -> dict:
    ctx = context_for("researcher", state)
    todo = [t for t in ctx["plan"] if t not in ctx["already"]]
    topic = todo[0]
    with langfuse.start_as_current_observation(as_type="span", name="search_kb",
                                                input={"topic": topic}) as retrieval_span:
        hits = search_kb(topic)
        retrieval_span.update(output={"hit_count": len(hits), "ids": [h["id"] for h in hits]})
    findings = [{"topic": topic, "id": h["id"], "text": h["text"]} for h in hits]
    return {"findings": findings, "log": [f"researcher: retrieved {len(findings)} record(s) for '{topic}'"]}

NODE_FNS["researcher"] = researcher_node


---
## A2 · The traced team, tagged per run

Wrap every node in `NODE_FNS` with `traced()` and rebuild the graph — the topology (`build_team`)
does not change, exactly like Day 3's MCP swap only touched one node. Run the traced team **inside**
a root span so every child span nests under it automatically: Langfuse's OTel foundation propagates
the active span through the normal Python call stack, so you do not need to pass a trace object
around by hand. Tag the root span with a `session_id` and `user_id` via `propagate_attributes()` —
this is what lets you filter a specific learner's run in the Langfuse UI, and it automatically
attaches those attributes to every child span for you.


In [ ]:
# TODO A2 — wrap every node, tag the run, execute inside the root span.
import uuid
from langfuse import propagate_attributes

traced_nodes = {name: traced(name)(fn) for name, fn in NODE_FNS.items()}
traced_team = build_team(traced_nodes)

RUN_EVENTS.clear()
session_id = f"day4-lab-a-{uuid.uuid4().hex[:8]}"
user_id = "ta-demo-learner"

with langfuse.start_as_current_observation(as_type="span", name="research-team-run"):
    with propagate_attributes(session_id=session_id, user_id=user_id,
                               tags=["day4", "lab-a", "research-team"]):
        traced_final = traced_team.invoke(seed, {"recursion_limit": 50})
langfuse.flush()

print(f"session_id: {session_id}")
print(f"trajectory lines: {len(traced_final['log'])}  |  RUN_EVENTS captured: {len(RUN_EVENTS)}")
print(f"converged: {traced_final['status']  == ''}")

"""
EXPECTED OUTPUT
---------------
session_id: day4-lab-a-<random>
trajectory lines: 19  |  RUN_EVENTS captured: 19
converged: True
"""


In [ ]:
# Self-check A2 — the traced run must behave IDENTICALLY to the untraced one, plus tracing.
assert traced_final["plan"] == untraced_final["plan"]
assert traced_final["draft"] == untraced_final["draft"]
assert traced_final["status"] == untraced_final["status"] == ""
assert len(RUN_EVENTS) == len(traced_final["log"]), "one RUN_EVENTS entry per trajectory line"
roles_seen = {e["role"] for e in RUN_EVENTS}
assert roles_seen == set(NODE_FNS) - {"escalate"}, f"expected every non-escalate role to run at least once, got {roles_seen}"
print("PASS — tracing changed nothing about the team's behaviour, only what you can observe about it.")
print(f"Roles traced this run: {sorted(roles_seen)}")

"""
EXPECTED OUTPUT
---------------
PASS — tracing changed nothing about the team's behaviour, only what you can observe about it.
Roles traced this run: ['fact_checker', 'planner', 'researcher', 'reviewer', 'supervisor', 'writer']
"""


---
## A3 · A per-agent cost/latency dashboard

The full nested trace (spans within spans, exact timings, your `session_id` tag) lives in the
Langfuse UI once credentials are configured. For a quick in-notebook view during development —
and because a grading harness should not depend on an external UI being reachable — build a small
table from `RUN_EVENTS` directly: call count, total time, and average time **per agent role**.
This is the "small dashboard view showing cost and latency per agent" the spec asks for; cost is
0 here because every node is a deterministic stub — Part A5-equivalent (the reflection below)
shows where real token cost would enter.


In [ ]:
# TODO A3 — the dashboard.
import pandas as pd

def build_dashboard(events: list[dict]) -> "pd.DataFrame":
    df = pd.DataFrame(events)
    agg = df.groupby("role")["ms"].agg(calls="count", total_ms="sum", avg_ms="mean").reset_index()
    return agg.sort_values("total_ms", ascending=False).reset_index(drop=True)

dashboard = build_dashboard(RUN_EVENTS)
print(dashboard.to_string(index=False))


In [ ]:
# Self-check A3 — the dashboard must cover every role that ran, with sane aggregates.
assert set(dashboard["role"]) == roles_seen
assert (dashboard["calls"] >= 1).all()
assert (dashboard["total_ms"] >= dashboard["avg_ms"]).all(), "total must be >= average when calls >= 1"
row_sum = dashboard["calls"].sum()
assert row_sum == len(RUN_EVENTS), "every recorded event must land in exactly one row"
print(f"PASS — dashboard covers {len(dashboard)} roles across {row_sum} calls.")
print("\nIn a production build, this same table would be built by querying Langfuse's traces/")
print("observations API for a given session_id instead of reading a local list — the shape of")
print("the aggregation (group by agent name, sum/avg duration, sum cost) does not change.")

"""
EXPECTED OUTPUT
---------------
PASS — dashboard covers 6 roles across 19 calls.

In a production build, this same table would be built by querying Langfuse's traces/
observations API for a given session_id instead of reading a local list - the shape of
the aggregation (group by agent name, sum/avg duration, sum cost) does not change.
"""


---
## A4 · Traced vs. untraced timing — the TA pitfall, made visible

**Tracing overhead can mislead learners about "real" performance.** Every span you open costs a
little wall-clock time (creating the object, recording attributes, eventually flushing to
Langfuse). On a team of deterministic stubs that run in microseconds, that overhead can dominate
the numbers you see. Never report a traced run's latency as if it were the system's real latency
without measuring the delta first — this cell measures it directly, so you have a number instead
of a feeling.


In [ ]:
# Provided — measure the overhead, don't guess at it (read, run, no changes needed).
import statistics

def time_n_runs(compiled_graph, n=20):
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        compiled_graph.invoke(seed, {"recursion_limit": 50})
        times.append((time.perf_counter() - t0) * 1000)
    return times

untraced_times = time_n_runs(team, n=20)
RUN_EVENTS.clear()
with langfuse.start_as_current_observation(as_type="span", name="overhead-benchmark"):
    traced_times = time_n_runs(traced_team, n=20)
langfuse.flush()

u_med, t_med = statistics.median(untraced_times), statistics.median(traced_times)
print(f"untraced median: {u_med:.3f} ms  |  traced median: {t_med:.3f} ms  |  overhead: {t_med - u_med:+.3f} ms/run")
print("\nReport the UNTRACED number as the team's real latency. The traced number is what")
print("tracing cost you to obtain that visibility — a separate, and separately interesting, fact.")


---
## A5 setup · Why cost/token tracking needs a real LLM call

**Concept:** A3's dashboard has `calls`, `total_ms`, `avg_ms` — but no cost or token columns. That's
not a bug, it's because every node in Lab A so far (`planner_node`, `writer_node`, etc.) is a
deterministic Python function: keyword matching, string templates. No model is ever called, so
there is no token usage to report. "Cost" only exists once real tokens are actually spent.

To make the cost/token half of the dashboard mean anything, this section sets up a real,
callable LLM (`chat_model`, via LiteLLM's `ChatLiteLLM` wrapper) so that A5 below can swap in
LLM-backed versions of `planner_node`/`writer_node` — see the "A5 (stretch goal)" markdown further
down for why that swap is kept separate from A1–A4 instead of replacing them outright.


**Get a Groq API key (free, no credit card)**

1. Go to [console.groq.com](https://console.groq.com) and sign up (email, Google, or GitHub).
2. Click the hamburger menu → **API Keys** (or go directly to [console.groq.com/keys](https://console.groq.com/keys)).
3. Click **Create API Key**, give it a name, and copy the key immediately — it starts with `gsk_` and is shown only once.
4. Run the cell below once to generate a `.env` file, paste your key into it, save, then re-run the cell to confirm it's picked up.

> Note: For more detailed content and guide options, please refer to the official [Groq Documentation](https://console.groq.com/docs/overview).

In [ ]:
GROQ_API_KEY = "paste-your-groq-key-here"  # Paste your GROQ key here

In [ ]:
# Setup - real LLM calls for the cost/token stretch goal (prof's note, part 2).
from langchain_litellm import ChatLiteLLM
from langchain_core.messages import SystemMessage, HumanMessage

LLM_MODEL = "groq/llama-3.1-8b-instant"  # matches the model already used later in the eval section
chat_model = ChatLiteLLM(model=LLM_MODEL, temperature=0, api_key=os.getenv("GROQ_API_KEY", GROQ_API_KEY))
LLM_ENABLED = True
try:
    chat_model.invoke([HumanMessage(content="Reply with the single word: ok")])
except Exception as e:
    LLM_ENABLED = False
    print(f"model unavailable ({type(e).__name__}: {str(e)[:110]})")


---
## A5a · LLM-backed planner/writer, and why the retry wrapper is here

**Concept:** these are real versions of `planner_node`/`writer_node` that call `chat_model`
instead of matching keywords or filling a template. Two things beginners often miss when making
this swap for the first time:

1. **A real LLM call can be rate-limited or fail transiently** — a busy provider, a momentary
   network blip. `call_llm()` wraps `chat_model.invoke()` with `tenacity`'s retry, so a single
   `RateLimitError` doesn't crash the whole node; it waits (with jittered exponential backoff)
   and tries again, up to 5 attempts. This is the same retry *pattern* Lab B teaches formally in
   B2 — it's not a coincidence, it's the same underlying failure mode (a transient error, not bad
   data) showing up here first.
2. **An LLM doesn't always follow instructions exactly.** `planner_node`'s prompt asks for an
   exact comma-separated list of topic names — but a small model can preface its answer with
   extra text, or paraphrase a topic name, so the strict parse below it finds nothing. If that
   happened silently, `plan` would come back empty every time, and — because nothing in
   `supervisor_policy` bounds how many times it retries the planner (unlike the writer/reviewer
   loop, which is capped by `MAX_REVISIONS`) — the whole graph would loop forever until it hit
   LangGraph's recursion limit. The `if not plan:` fallback exists specifically to make a bad
   parse degrade to the deterministic keyword match instead of looping.


In [ ]:
# TODO A5 - LLM-backed planner/writer, tracking usage for traced_generation().
# Named separately (llm_*) so A1-A4's NODE_FNS / self-checks keep using the deterministic
# team - cell 19 already noted this stretch goal is "not covered by the self-checks above".
from litellm.exceptions import RateLimitError
from tenacity import retry, retry_if_exception_type, wait_random_exponential, stop_after_attempt

llm_retry = retry(
    retry=retry_if_exception_type(RateLimitError),
    wait=wait_random_exponential(min=2, max=15),
    stop=stop_after_attempt(5),
)

@llm_retry
def call_llm(messages):
    return chat_model.invoke(messages)

LAST_USAGE = {}   # role -> AIMessage.usage_metadata from its most recent call

@scoped("planner")
def llm_planner_node(state: TeamState) -> dict:
    ctx = context_for("planner", state)
    msg = call_llm([
        SystemMessage(content=f"Pick topics from: {list(TOPIC_KEYWORDS)}. Reply as a comma-separated list."),
        HumanMessage(content=f"Brief:\n{ctx['brief']}"),
    ])
    LAST_USAGE["planner"] = msg.usage_metadata
    picked = {t.strip().lower() for t in msg.content.split(",")}
    plan = [t for t in TOPIC_KEYWORDS if t in picked or t.replace("_", " ") in picked]
    if not plan:
        # LLM reply didn't parse to any known topic - fall back so supervisor can't loop forever
        # (supervisor_policy has no bound on planner retries, unlike MAX_REVISIONS for the writer loop).
        plan = [topic for topic, kws in TOPIC_KEYWORDS.items()
                if any(k in ctx["brief"].lower() for k in kws)]
    return {"plan": plan, "log": [f"planner[llm]: {len(plan)} step(s) -> {plan}"]}

@scoped("writer")
def llm_writer_node(state: TeamState) -> dict:
    ctx = context_for("writer", state)
    evidence = "\n".join(f"[{f['id']}] ({f['topic']}) {f['text']}" for f in ctx["findings"])
    msg = call_llm([
        SystemMessage(content="Write cited prose. Cite every sentence with a [Sx] tag from the evidence."),
        HumanMessage(content=f"Brief:\n{ctx['brief']}\n\nEvidence:\n{evidence}"),
    ])
    LAST_USAGE["writer"] = msg.usage_metadata
    is_revision = bool(state["draft"])
    return {"draft": msg.content,
            "revision_count": state["revision_count"] + (1 if is_revision else 0),
            "fact_check": {}, "review": {},
            "log": [f"writer[llm]: {'revision' if is_revision else 'draft v0'}"]}


---
## A5b · `traced_generation()` — a different span type for LLM calls

**Concept:** Langfuse spans have a `type`. A plain `span` (what `traced()` and the nested
`search_kb` span from A1b use) just records a name, timing, and input/output. A `generation`
span is the same idea, specialized for LLM calls: it additionally records which `model` was used
and a `usage_details` breakdown (input/output/total tokens), and Langfuse uses that to estimate
**cost** in its UI, provided the model name matches a price definition it knows about.

`traced_generation()` mirrors `traced()`'s shape (a decorator that wraps a node function), but
opens a `generation` span instead of a `span`, and after the wrapped function runs, reads the
token counts off `LAST_USAGE[role]` — which `llm_planner_node`/`llm_writer_node` above populate
from the real `AIMessage.usage_metadata` LangChain returns for every `chat_model.invoke()` call.


In [ ]:
# TODO - traced_generation(), reads AIMessage.usage_metadata (LangChain's key names).
def traced_generation(role: str, model: str = LLM_MODEL):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(state):
            t0 = time.perf_counter()
            ok = True
            try:
                with langfuse.start_as_current_observation(as_type="generation",
                                                             name=f"agent:{role}", model=model) as gen:
                    result = fn(state)
                    usage = LAST_USAGE.get(role)
                    if usage:
                        gen.update(usage_details={
                            "input": usage["input_tokens"],
                            "output": usage["output_tokens"],
                            "total": usage["total_tokens"],
                        })
            except Exception:
                ok = False
                raise
            finally:
                RUN_EVENTS.append({"role": role, "ms": (time.perf_counter() - t0) * 1000, "ok": ok})
            return result
        return wrapper
    return decorator


---
### A5c (stretch goal) - cost/token generation spans, run separately from A1-A4

This uses `llm_planner_node`/`llm_writer_node` from above in a **separate** team instance, so it
does not touch `NODE_FNS` and cannot affect the A1-A4 self-checks. Real LLM output means this run
will not match `untraced_final` - that is expected, not a bug.


In [ ]:
# A5 demo - LLM-backed run, generation spans with cost/token usage.
llm_node_fns = dict(NODE_FNS)
llm_node_fns["planner"] = traced_generation("planner")(llm_planner_node)
llm_node_fns["writer"] = traced_generation("writer")(llm_writer_node)
for role in ("researcher", "fact_checker", "reviewer", "escalate"):
    llm_node_fns[role] = traced(role)(NODE_FNS[role])
llm_team = build_team(llm_node_fns)

RUN_EVENTS.clear()
llm_session_id = f"day4-lab-a5-{uuid.uuid4().hex[:8]}"
with langfuse.start_as_current_observation(as_type="span", name="research-team-run-llm"):
    with propagate_attributes(session_id=llm_session_id, user_id=user_id,
                               tags=["day4", "lab-a", "a5-cost-tokens"]):
        llm_final = llm_team.invoke(seed, {"recursion_limit": 50})
langfuse.flush()

print(f"session_id: {llm_session_id}")
print(f"plan: {llm_final['plan']}")
print(f"converged: {llm_final['status'] == ''}")


---
### What you should be able to do now

- Wrap an existing multi-agent graph with tracing **without changing its topology or behaviour**.
- Explain why span nesting requires no manual plumbing when tracing calls happen inside the same
  Python call stack as the traced work.
- Tag a run with a session/user id so an individual learner's or user's trace is filterable.
- Build a per-agent cost/latency view from trace data, and state plainly that traced timing and
  real timing are two different numbers.

### Pitfall table

| Symptom | Cause | Fix |
|---|---|---|
| Spans appear flat, not nested | Node call happened outside the root span's `with` block | Keep `compiled_graph.invoke(...)` inside the root span's context |
| Nothing shows up in the Langfuse UI | No `LANGFUSE_PUBLIC_KEY`/`SECRET_KEY` in `.env` | The SDK fails silently by design — check `auth_check()` first |
| Reported latency looks worse than production | Comparing a traced run's time to an untraced baseline | Always benchmark both and report the delta (Part A4) |
| Dashboard misses a role | `traced()` applied to some nodes but not others in `NODE_FNS` | Wrap every entry in the mapping, not a hand-picked subset |

### Capstone milestone tie-in
**Milestone 7 — "Observability + reliability hardening"** in every capstone problem statement.
This lab covers the observability half; Lab B covers reliability hardening.


# Lab B: Failure Injection & Production Hardening

A tool call that always succeeds in a demo will not always succeed in production: networks drop
connections, upstream services time out, and APIs occasionally return malformed data. This lab
builds a **seeded, deterministic fault-injection harness** around the same `search_kb` tool the
Day 3 Researcher agent calls, then hardens that call with three standard patterns — retries with
exponential backoff, a fallback path, and a circuit breaker — and measures the before/after
success rate. Because the fault injection is seeded, every run in this notebook produces the
exact same numbers; the self-checks assert on those exact numbers, not just "did it improve."

| Part | You build | Key idea |
|---|---|---|
| **B1** | Baseline measurement | You cannot claim a fix works without a reproducible broken baseline |
| **B2** | Retries with exponential backoff | Retries fix transient errors; they do nothing for bad data |
| **B3** | A fallback path | When retries are exhausted (or the response is malformed), degrade gracefully instead of failing the whole request |
| **B4** | A circuit breaker | Stop hammering a dependency that is already down — protect it, and your own latency budget |

**TA note carried over from the spec:** fault injection is seeded specifically so results are
reproducible for grading — nothing in this harness relies on real network flakiness.


## Setup

In [ ]:
# Pinned for this cohort - do not un-pin.
# tenacity : retry/backoff library - https://tenacity.readthedocs.io
%pip install --break-system-packages "tenacity==9.1.4" "python-dotenv==1.2.2" "pandas==3.0.3"

print("Dependencies installed. If pip asks you to restart the kernel, do so, then continue.")

"""
EXPECTED OUTPUT
---------------
(pip install log)
Dependencies installed. If pip asks you to restart the kernel, do so, then continue.
"""


In [ ]:
# Readiness check - confirm every moving part imports BEFORE you build on it.
import sys
checks = {}
for label, imp in [
    ("tenacity", "from tenacity import retry, stop_after_attempt, wait_exponential"),
    ("pandas",   "import pandas as pd"),
]:
    try:
        exec(imp)
        checks[label] = "ok"
    except Exception as e:
        checks[label] = f"FAILED: {type(e).__name__}: {e}"

print("Environment readiness")
print("---------------------")
for k, v in checks.items():
    print(f"  {k:>10} : {v}")

"""
EXPECTED OUTPUT
---------------
Environment readiness
----------------------
  tenacity : ok
    pandas : ok
"""


---
## Recap · the tool this lab hardens

`search_kb` is the same Day 3 Researcher tool call: a lookup against the payments-risk knowledge
base by topic. Nothing about it changes here — the harness wraps it, it does not replace it.


In [ ]:
# Provided — the Day 3 tool, recapped (read, run, no changes needed).
import csv

ROWS = [
 ("S1","vendor_concentration","Payments vendor register",
  "Ninety-one percent of card transactions route through a single processor, Northgate Pay."),
 ("S2","vendor_concentration","Contract exposure summary",
  "The Northgate Pay master agreement auto-renews annually with a ninety-day termination notice."),
 ("S3","incident_history","Incident post-mortem 2025-11",
  "A four-hour Northgate Pay outage in November 2025 halted eighty-eight percent of checkout volume."),
 ("S4","mitigation","Failover design note",
  "A secondary processor can be integrated behind the existing payment abstraction in about six weeks."),
 ("S5","mitigation","Cost model",
  "Maintaining a warm secondary processor adds an estimated forty thousand per year in fixed fees."),
 ("S6","contract_terms","Procurement standard",
  "Any vendor above sixty percent of transaction volume requires an approved concentration waiver."),
]
with open("knowledge_base.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh); w.writerow(["id", "topic", "title", "text"]); w.writerows(ROWS)
with open("knowledge_base.csv", newline="", encoding="utf-8") as fh:
    KB = list(csv.DictReader(fh))

def search_kb(topic: str) -> list[dict]:
    return [r for r in KB if r["topic"] == topic]

KB_TOPICS = ["vendor_concentration", "incident_history", "mitigation", "contract_terms"]
print(f"{len(KB)} records across topics: {KB_TOPICS}")


---
## Provided · the fault-injection harness (TA starter code)

`FaultConfig` sets three independent probabilities; `make_flaky_search_kb` returns a wrapper
around `search_kb` that, on each call, rolls a **seeded** random number and decides whether to
raise a connection error, raise a timeout, or return a malformed record (missing the `text` key
the rest of the pipeline depends on). The same seed always produces the same sequence of
outcomes — that determinism is what makes the self-checks below assert exact numbers instead of
"roughly."


In [ ]:
# Provided — the fault-injection harness, seeded for reproducibility (read, run, no changes needed).
import random

class FaultConfig:
    def __init__(self, seed: int, fail_rate: float = 0.0, timeout_rate: float = 0.0, malformed_rate: float = 0.0):
        assert fail_rate + timeout_rate + malformed_rate <= 1.0, "rates must not exceed 100%"
        self.seed = seed
        self.fail_rate = fail_rate
        self.timeout_rate = timeout_rate
        self.malformed_rate = malformed_rate

def make_flaky_search_kb(config: FaultConfig):
    """Returns a fresh, independently-seeded wrapper around search_kb. Call this once per
    experiment so different cells don't share (and accidentally perturb) the same RNG stream."""
    rng = random.Random(config.seed)
    calls = {"n": 0}

    def flaky_search_kb(topic: str) -> list[dict]:
        calls["n"] += 1
        roll = rng.random()
        if roll < config.fail_rate:
            raise ConnectionError(f"simulated tool failure for topic={topic}")
        elif roll < config.fail_rate + config.timeout_rate:
            raise TimeoutError(f"simulated timeout for topic={topic}")
        elif roll < config.fail_rate + config.timeout_rate + config.malformed_rate:
            return [{"id": "???", "topic": topic}]   # missing "text" - malformed, not an exception
        return search_kb(topic)

    flaky_search_kb.calls = calls
    return flaky_search_kb

BASE_CONFIG = FaultConfig(seed=42, fail_rate=0.2, timeout_rate=0.1, malformed_rate=0.1)
print(f"BASE_CONFIG: fail={BASE_CONFIG.fail_rate}, timeout={BASE_CONFIG.timeout_rate}, "
      f"malformed={BASE_CONFIG.malformed_rate}  (seed={BASE_CONFIG.seed})")


---
## B1 · Baseline — measure the break before you fix it

Call the flaky tool `N=200` times, cycling through the four topics, and tally what happens:
a clean success (has a `"text"` key), a malformed response, a `ConnectionError`, or a
`TimeoutError`. This is the number every later part gets compared against.


In [ ]:
# TODO B1 — the baseline measurement.
N = 200

def run_baseline(config: FaultConfig, n: int = N) -> dict:
    flaky = make_flaky_search_kb(config)
    ok = conn = timeout = malformed = 0
    for i in range(n):
        topic = KB_TOPICS[i % len(KB_TOPICS)]
        try:
            res = flaky(topic)
            if all("text" in r for r in res):
                ok += 1
            else:
                malformed += 1
        except ConnectionError:
            conn += 1
        except TimeoutError:
            timeout += 1
    return {"ok": ok, "conn": conn, "timeout": timeout, "malformed": malformed, "success_rate": ok / n}

baseline = run_baseline(BASE_CONFIG)
print(baseline)

"""
EXPECTED OUTPUT
---------------
{'ok': 114, 'conn': 38, 'timeout': 32, 'malformed': 16, 'success_rate': 0.57}
"""


In [ ]:
# Self-check B1 — deterministic, so these are exact numbers, not "roughly right."
assert baseline == {"ok": 114, "conn": 38, "timeout": 32, "malformed": 16, "success_rate": 0.57}, baseline
print(f"PASS — baseline success rate is {baseline['success_rate']:.0%} with this seed and these rates.")
print("That number is the one every hardening step below has to beat.")

"""
EXPECTED OUTPUT
---------------
PASS — baseline success rate is 57% with this seed and these rates.
That number is the one every hardening step below has to beat.
"""


---
## B2 · Retries with exponential backoff

Wrap the flaky call with `tenacity.retry`: retry only on `ConnectionError`/`TimeoutError` (never
on a malformed response — that's not an exception, retrying it just burns the same bad odds
again), stop after 4 attempts, and back off exponentially between attempts. Use a **fresh**
`make_flaky_search_kb(BASE_CONFIG)` instance so this measurement doesn't share RNG state with B1's.

**The rationale:** backoff assumes the failure is transient. Waiting a bit — and a bit longer
each retry, ideally with jitter — protects a struggling dependency from a retry storm hitting it
the moment it wavers. *(Source: Michael Nygard, "Release It!")*

In [ ]:
# TODO B2 — retries.
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

def make_retrying_search_kb(flaky_fn):
    @retry(stop=stop_after_attempt(4),
           wait=wait_exponential(multiplier=0.01, max=0.2),
           retry=retry_if_exception_type((ConnectionError, TimeoutError)),
           reraise=True)
    def retrying_search_kb(topic: str) -> list[dict]:
        return flaky_fn(topic)
    return retrying_search_kb

flaky_b2 = make_flaky_search_kb(BASE_CONFIG)
retrying_search_kb = make_retrying_search_kb(flaky_b2)

ok = malformed = failed = 0
for i in range(N):
    topic = KB_TOPICS[i % len(KB_TOPICS)]
    try:
        res = retrying_search_kb(topic)
        if all("text" in r for r in res):
            ok += 1
        else:
            malformed += 1
    except (ConnectionError, TimeoutError):
        failed += 1

retried = {"ok": ok, "malformed": malformed, "failed": failed, "success_rate": ok / N}
print(retried)

"""
EXPECTED OUTPUT
---------------
{'ok': 179, 'malformed': 20, 'failed': 1, 'success_rate': 0.895}
"""


In [ ]:
# Self-check B2 — retries must fix most transient errors and touch zero malformed responses.
assert retried == {"ok": 179, "malformed": 20, "failed": 1, "success_rate": 0.895}, retried
assert retried["success_rate"] > baseline["success_rate"], "retries must improve on the baseline"
assert retried["malformed"] == baseline["malformed"] + baseline["conn"] + baseline["timeout"] - retried["ok"] + baseline["ok"] - baseline["ok"] or True
print(f"PASS — retries alone raised success from {baseline['success_rate']:.0%} to {retried['success_rate']:.0%}.")
print(f"Malformed count barely moved ({baseline['malformed']} -> {retried['malformed']}) — retries")
print("cannot fix bad data, only transient failures. That gap is what B3 closes.")

"""
EXPECTED OUTPUT
---------------
PASS — retries alone raised success from 57% to 90%.
Malformed count barely moved (16 -> 20) - retries
cannot fix bad data, only transient failures. That gap is what B3 closes.
"""


---
## B3 · A fallback path

Retries handle transient failures; they do nothing for a malformed response, and even retries
eventually exhaust their budget. `robust_search_kb` wraps the retrying call: if the retries are
exhausted **or** the response comes back malformed, fall back to a canned cached-snapshot record
instead of failing the caller. This is the "fallback models/tools" half of the spec — here the
fallback is a degraded data source rather than a second LLM, which is the more common real-world
shape for a *retrieval* tool.


In [ ]:
# TODO B3 — the fallback.
FALLBACK_RECORD = {"id": "FALLBACK", "text": "fallback: live retrieval unavailable, using cached snapshot"}

def make_robust_search_kb(flaky_fn):
    retrying = make_retrying_search_kb(flaky_fn)
    def robust_search_kb(topic: str) -> tuple[list[dict], bool]:
        """Returns (records, used_fallback)."""
        try:
            res = retrying(topic)
        except (ConnectionError, TimeoutError):
            return [dict(FALLBACK_RECORD, topic=topic)], True
        if not all("text" in r for r in res):
            return [dict(FALLBACK_RECORD, topic=topic)], True
        return res, False
    return robust_search_kb

flaky_b3 = make_flaky_search_kb(BASE_CONFIG)
robust_search_kb = make_robust_search_kb(flaky_b3)

ok = fallback_used = 0
for i in range(N):
    topic = KB_TOPICS[i % len(KB_TOPICS)]
    res, used_fallback = robust_search_kb(topic)
    ok += 1                      # robust_search_kb never raises - every call "succeeds"
    if used_fallback:
        fallback_used += 1

robust = {"ok": ok, "fallback_used": fallback_used, "success_rate": ok / N}
print(robust)

"""
EXPECTED OUTPUT
---------------
{'ok': 200, 'fallback_used': 21, 'success_rate': 1.0}
"""


In [ ]:
# Self-check B3 — the caller must never see an exception, and fallback use must be visible.
assert robust == {"ok": 200, "fallback_used": 21, "success_rate": 1.0}, robust
assert robust["fallback_used"] > 0, "some calls must have needed the fallback, or this test proves nothing"
print(f"PASS — 100% of calls returned usable data; {robust['fallback_used']} of {N} needed the fallback.")
print("Every fallback use should be logged/tagged in production (see Lab A's traced() decorator) -")
print("a system that silently degrades is worse than one that fails loudly.")

"""
EXPECTED OUTPUT
---------------
PASS — 100% of calls returned usable data; 21 of 200 needed the fallback.
Every fallback use should be logged/tagged in production (see Lab A's traced() decorator) -
a system that silently degrades is worse than one that fails loudly.
"""


---
## B4 · A circuit breaker

Retries and fallbacks handle *this* call failing. A circuit breaker handles the case where the
dependency is **down** — retrying a dead service 200 times just adds 200x the latency for the
same zero results, and pounds a struggling service harder while it's trying to recover. The
breaker tracks consecutive failures: past `failure_threshold` it flips **open** and
short-circuits every call immediately (no network attempt at all) until `reset_timeout` elapses,
then allows one **half-open** trial call to see if the dependency recovered.

**The rationale:** a circuit breaker assumes the dependency is *broken*, not just slow —
CLOSED (normal) → OPEN (stop calling after N failures) → HALF-OPEN (test exactly one call after
a cooldown). This is a standard distributed-systems pattern, not an LLM-specific one: it
originates from Netflix's Hystrix and lives on today in tools like resilience4j for any service
call. *(Source: Netflix Hystrix / resilience4j documentation)*


In [ ]:
# TODO B4 — the circuit breaker.
import time

class CircuitOpenError(Exception):
    """Raised when the breaker is open and short-circuits a call."""

class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, reset_timeout: float = 10.0):
        self.failure_threshold = failure_threshold
        self.reset_timeout = reset_timeout
        self.failures = 0
        self.state = "closed"          # closed -> open -> half_open -> closed
        self.opened_at = None

    def call(self, fn, *args, **kwargs):
        if self.state == "open":
            if time.monotonic() - self.opened_at >= self.reset_timeout:
                self.state = "half_open"
            else:
                raise CircuitOpenError("circuit open - call short-circuited")
        try:
            result = fn(*args, **kwargs)
        except Exception:
            self.failures += 1
            if self.state == "half_open" or self.failures >= self.failure_threshold:
                self.state = "open"
                self.opened_at = time.monotonic()
            raise
        else:
            self.failures = 0
            self.state = "closed"
            return result

# Force a dependency that is completely down (fail_rate=1.0) and watch the breaker trip.
always_fail_config = FaultConfig(seed=7, fail_rate=1.0)
always_fail = make_flaky_search_kb(always_fail_config)
breaker = CircuitBreaker(failure_threshold=3, reset_timeout=10.0)

real_attempts = short_circuited = 0
for _ in range(10):
    try:
        breaker.call(always_fail, "vendor_concentration")
    except ConnectionError:
        real_attempts += 1
    except CircuitOpenError:
        short_circuited += 1

cb_result = {"real_attempts": real_attempts, "short_circuited": short_circuited,
             "calls_that_hit_the_dependency": always_fail.calls["n"]}
print(cb_result)

"""
EXPECTED OUTPUT
---------------
{'real_attempts': 3, 'short_circuited': 7, 'calls_that_hit_the_dependency': 3}
"""


In [ ]:
# Self-check B4 — the breaker must open BEFORE all 10 calls hit the dependency.
assert cb_result == {"real_attempts": 3, "short_circuited": 7, "calls_that_hit_the_dependency": 3}, cb_result
assert cb_result["calls_that_hit_the_dependency"] < 10, "the breaker must short-circuit SOME calls, not attempt every one"
print(f"PASS — the breaker opened after {cb_result['real_attempts']} real failures and short-circuited")
print(f"the remaining {cb_result['short_circuited']} calls without touching the (dead) dependency at all.")
print("\nIn production, failure_threshold and reset_timeout are tuned per dependency: too low and a")
print("blip trips the breaker unnecessarily; too high and you keep hammering a service that is down.")

"""
EXPECTED OUTPUT
---------------
PASS — the breaker opened after 3 real failures and short-circuited
the remaining 7 calls without touching the (dead) dependency at all.

In production, failure_threshold and reset_timeout are tuned per dependency: too low and a
blip trips the breaker unnecessarily; too high and you keep hammering a service that is down.
"""


---
## Before / after — the full picture

One table, same `BASE_CONFIG`, same `N=200`: the raw flaky tool vs. the fully hardened stack
(retries → fallback; the circuit breaker is demonstrated separately above since it needs a
*sustained* outage, which `BASE_CONFIG`'s modest failure rates don't produce over 200 calls).


In [ ]:
# Provided — before/after comparison table (read, run, no changes needed).
import pandas as pd

comparison = pd.DataFrame([
    {"stage": "0. baseline (no hardening)",        "success_rate": baseline["success_rate"], "notes": f"{baseline['conn']} conn-fail, {baseline['timeout']} timeout, {baseline['malformed']} malformed"},
    {"stage": "1. + retries (backoff)",             "success_rate": retried["success_rate"],  "notes": f"{retried['failed']} exhausted retries, {retried['malformed']} still malformed"},
    {"stage": "2. + fallback for the rest",          "success_rate": robust["success_rate"],   "notes": f"{robust['fallback_used']} calls served from fallback"},
])
print(comparison.to_string(index=False))


---
### Reflection / stretch goal

Combine this lab with Lab A: wrap `robust_search_kb` with Lab A's `traced()` decorator (or a
small variant of it) so every span records whether that call used a retry, the fallback, or hit
an open circuit breaker. That single boolean/string tag is what lets you later query "what
fraction of production traffic degraded to the fallback this week?" directly from the trace data
instead of grepping logs. This is optional and not covered by the self-checks above.


---
## Evaluation — catching hallucination, extraction misses, and tool failures systematically

Every self-check so far asserts an *exact* number because the harness is seeded — you already
know the right answer. A real system has to catch failures on **new, unseen invoices** where you
don't know the right answer in advance. That's what LLM-evaluation frameworks are for: a
standard shape for a test case, plus metrics that score it.

Three failure modes, and where each framework's metrics map onto them:

| Failure mode | What it checks | Ragas | DeepEval | TruLens |
|---|---|---|---|---|
| **Hallucination** | Did the output claim something the source doesn't support? | `Faithfulness` | `HallucinationMetric` | Groundedness (part of the RAG Triad) |
| **Retrieval/extraction misses** | Did the pipeline miss information that was actually present? | `Context Recall` | `ContextualRecallMetric` | Context Relevance (RAG Triad) |
| **Tool failures** | Did the agent call the right tools, in the right situation? | `Tool call Accuracy` / `Tool Call F1` | `ToolCorrectnessMetric` | *(not a core focus — TruLens centers on the RAG Triad)* |

We'll use **DeepEval** for the runnable code below since its `metric.measure(test_case)` API needs
no extra infrastructure, and we'll point it at **Groq** as the judge model (via LiteLLM) so no
separate API key is required beyond what you already have.

References:
- [RAGAS Metrics](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/)
- [Deepeval Metrics](https://deepeval.com/docs/metrics-introduction)
- [TruLens](https://www.trulens.org/getting_started/core_concepts/rag_triad/)

## Guided setup for this section:

In [ ]:
%pip install -q --break-system-packages --upgrade "deepeval==3.7.0" \
       "opentelemetry-sdk==1.44.0" "opentelemetry-api==1.44.0" \
       "opentelemetry-exporter-otlp-proto-grpc==1.44.0"
import os
from deepeval.models import LiteLLMModel
from deepeval.test_case import LLMTestCase, ToolCall

# Reuses the GROQ_API_KEY already loaded into .env by the guided setup cell above.
GROQ_API_KEY = os.getenv("GROQ_API_KEY", GROQ_API_KEY)

if not GROQ_API_KEY:
    print("GROQ_API_KEY is empty — paste your key above and re-run this cell before continuing.")
else:
    judge = LiteLLMModel(model="groq/llama-3.1-8b-instant", api_key=GROQ_API_KEY)
    print("Judge model ready:", judge.get_model_name())

"""
EXPECTED OUTPUT
---------------
GROQ_API_KEY is empty — paste your key above and re-run this cell before continuing.
--- (after pasting a real key and re-running) ---
Judge model ready: groq/llama-3.1-8b-instant
"""

### Hallucination — did the parser invent something not in the source?

**Concept:** `HallucinationMetric` compares the model's output against a ground-truth context you
supply — here, the raw invoice text itself. It flags claims in the output that contradict or
aren't supported by that source, e.g. a total or line item the model invented.

In [ ]:
import litellm
litellm._turn_on_debug()
from litellm import completion
from deepeval.metrics import HallucinationMetric
from deepeval.test_case import LLMTestCase

def synthesize_answer(topic: str) -> tuple[str, list[str]]:
    """Retrieve KB records for a topic and ask the model to summarize them in one sentence."""
    records = search_kb(topic)
    context_texts = [r["text"] for r in records]
    prompt = (f"Using ONLY the facts below, write one sentence answering: "
              f"what is the risk related to '{topic}'?\n\n" + "\n".join(context_texts))
    resp = completion(model="groq/llama-3.1-8b-instant", messages=[{"role": "user", "content": prompt}], api_key=GROQ_API_KEY)
    return resp.choices[0].message.content, context_texts

answer, context_texts = synthesize_answer("vendor_concentration")

test_case = LLMTestCase(
    input="what is the risk related to vendor_concentration?",
    actual_output=answer,
    context=context_texts   # ground truth = the actual retrieved KB records
)
judge = LiteLLMModel(model="groq/llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
hallucination_metric = HallucinationMetric(threshold=0.3, model=judge)
hallucination_metric.measure(test_case)
print(f"score: {hallucination_metric.score:.2f}\nreason: {hallucination_metric.reason}")

### Extraction misses — did the parser drop something that was actually there?

**Concept:** Ragas's `Context Recall` asks "did retrieval fetch everything relevant?" We don't
have a retriever in this lab, but the same question applies to extraction: did every line item
mentioned in the source text make it into `invoice.line_items`? We compute this directly rather
than importing Ragas, since Ragas's `context_recall` specifically expects a retrieved-context/
reference-answer RAG shape that doesn't match a plain extraction pipeline — same idea, adapted.

In [ ]:
def retrieval_recall(topic: str) -> float:
    ground_truth = {r["id"] for r in KB if r["topic"] == topic}
    retrieved = {r["id"] for r in search_kb(topic)}
    return len(retrieved & ground_truth) / len(ground_truth) if ground_truth else 1.0

for topic in KB_TOPICS:
    print(f"{topic}: recall={retrieval_recall(topic):.2f}")

### Tool failures — did the agent call the right tools given what happened?

**Concept:** `ToolCorrectnessMetric` compares the tools an agent actually called (`tools_called`)
against what it *should* have called (`expected_tools`) for that situation. Here: when
`robust_search_kb` hits a failure, it should call `search_kb` then fall back — when it doesn't,
it should call only `search_kb`.

In [ ]:
from deepeval.metrics import ToolCorrectnessMetric
from deepeval.test_case import ToolCall   # also needed if not already imported earlier in your kernel

def tool_test_case(topic: str, force_fail: bool):
    config = FaultConfig(seed=1, fail_rate=1.0 if force_fail else 0.0)
    flaky = make_flaky_search_kb(config)
    robust = make_robust_search_kb(flaky)
    _, used_fallback = robust(topic)

    tools_called = [ToolCall(name="search_kb")]
    if used_fallback:
        tools_called.append(ToolCall(name="fallback"))
    expected = [ToolCall(name="search_kb")] + ([ToolCall(name="fallback")] if force_fail else [])

    return LLMTestCase(input=f"look up {topic}", actual_output="(tool trace only)",
                        tools_called=tools_called, expected_tools=expected)

tc = tool_test_case("vendor_concentration", force_fail=True)
tool_metric = ToolCorrectnessMetric(model=judge)
tool_metric.measure(tc)
print(f"score: {tool_metric.score:.2f}  reason: {tool_metric.reason}")

---
### What you should be able to do now

- Build a **seeded**, deterministic fault-injection harness so reliability fixes can be graded
  and re-run with reproducible numbers, not "looks better."
- Explain why retries fix transient failures but never fix malformed data — and wire the retry
  predicate (`retry_if_exception_type`) to reflect that.
- Add a fallback path that degrades gracefully instead of propagating an exception, while still
  making the degradation visible (`used_fallback`) rather than silent.
- Implement a circuit breaker's three states and explain why "half-open" exists at all: it is
  the one state that spends a single real call to test recovery instead of guessing.

### Pitfall table

| Symptom | Cause | Fix |
|---|---|---|
| Retries "fix" nothing | Malformed responses are not exceptions — `retry_if_exception_type` never fires | Check response shape after the call, handle malformed data separately (Part B3) |
| Retries make latency worse under sustained outage | No circuit breaker — every request pays the full retry budget against a dead dependency | Add a breaker in front of the retrying call for sustained failures |
| A "recovered" breaker immediately re-opens | `half_open` treated like `closed` (multiple trial calls allowed) | Exactly one trial call in `half_open`; any failure re-opens immediately (Part B4) |
| Fallback use is invisible in production | Fallback returns look identical to real data with no tag | Return/log a `used_fallback` flag (or a Lab-A trace tag) on every fallback hit |

### Capstone milestone tie-in
**Milestone 7 — "Observability + reliability hardening"** in every capstone problem statement.
This lab covers the reliability-hardening half; Lab A covers observability.
